# BioNNE-R 2026 — Improved Baseline

**Improvements over the original OpenNRE baseline:**
1. Domain-specific backbone (BiomedBERT for EN, mDeBERTa-v3-base for RU/bilingual)
2. Typed entity markers `<H:TYPE>` / `<T:TYPE>` + entity start-marker representations
3. Nesting flag feature (binary, concatenated before classifier head)
4. Inverse-frequency class-weighted CrossEntropyLoss

**Runtime:** Connect this notebook to a Colab T4 GPU via the IDE extension before running.

## 0. Configuration — edit before running

In [19]:
TASK_DIR = "../"

In [25]:
# ── Repository ────────────────────────────────────────────────────────────
REPO_URL    = "https://github.com/nerel-ds/NEREL-BIO.git"   # change to your fork
REPO_DIR    = "/content/NEREL-BIO"
REPO_DIR    = "/home/obi/competitions/NEREL-BIO"
TASK_DIR    = f"{REPO_DIR}/BioNNE-R"
BASELINE    = f"{TASK_DIR}/baseline"

# ── Backbone models ───────────────────────────────────────────────────────
MODEL_EN    = "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext"
MODEL_RU    = "microsoft/mdeberta-v3-base"
MODEL_BILI  = "microsoft/mdeberta-v3-base"   # same as RU; bilingual uses combined data

# ── Training hyper-parameters ─────────────────────────────────────────────
BATCH_SIZE  = 16
LR          = 2e-5
EPOCHS      = 10
WARMUP      = 300
MAX_LEN     = 256
NEG_RATIO   = 3    # passed to prepare_data.py  (positive:negative = 1:NEG_RATIO)
SEED        = 42

# ── Resume ────────────────────────────────────────────────────────────────
# Set RESUME = True when restarting after a Colab interruption.
# Copy <model>.pt AND <model>.pt.resume from Drive back into outputs/ first,
# then run the relevant training cell — it will continue from the last
# completed epoch with the exact same LR schedule.
# Set RESUME = False to start fresh from HuggingFace weights.
RESUME      = False

# ── Google Drive ──────────────────────────────────────────────────────────
GDRIVE_DIR  = "/content/drive/MyDrive/BioNNE-R"   # folder created automatically

## 1. Environment setup

In [ ]:
!pip install -q transformers==4.40.0 torch pandas scikit-learn nltk tqdm

In [26]:
import os
import sys

In [ ]:
if not os.path.exists(REPO_DIR):
    !git clone --depth 1 {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

In [5]:
# Add baseline/ to sys.path so imports work
sys.path.insert(0, BASELINE)
os.chdir(BASELINE)
print("Working directory:", os.getcwd())


FileNotFoundError: [Errno 2] No such file or directory: '/content/NEREL-BIO/BioNNE-R/baseline'

In [ ]:
# Mount Google Drive early so checkpoints are safe throughout training
from google.colab import drive
drive.mount("/content/drive")
os.makedirs(GDRIVE_DIR, exist_ok=True)
print("Drive mounted. Checkpoints will be mirrored to:", GDRIVE_DIR)

## 2. Data preparation

In [27]:
import os
os.makedirs("data", exist_ok=True)
os.makedirs("outputs", exist_ok=True)

EN_TRAIN_REL = f"{TASK_DIR}/data/en/train/eng-train-rel.tsv"
EN_TRAIN_ENT = f"{TASK_DIR}/data/en/train/eng-train-ent.tsv"
EN_TRAIN_TXT = f"{TASK_DIR}/data/en/train/texts/"

EN_DEV_REL   = f"{TASK_DIR}/data/en/dev/eng-dev-rel.tsv"
EN_DEV_ENT   = f"{TASK_DIR}/data/en/dev/eng-dev-ent.tsv"
EN_DEV_TXT   = f"{TASK_DIR}/data/en/dev/texts/"

RU_TRAIN_REL = f"{TASK_DIR}/data/ru/train/rus-train-rel.tsv"
RU_TRAIN_ENT = f"{TASK_DIR}/data/ru/train/rus-train-ent.tsv"
RU_TRAIN_TXT = f"{TASK_DIR}/data/ru/train/texts/"

RU_DEV_REL   = f"{TASK_DIR}/data/ru/dev/rus-dev-rel.tsv"
RU_DEV_ENT   = f"{TASK_DIR}/data/ru/dev/rus-dev-ent.tsv"
RU_DEV_TXT   = f"{TASK_DIR}/data/ru/dev/texts/"

In [ ]:
# English — training split (with negative sampling)
!python prepare_data.py {EN_TRAIN_REL} {EN_TRAIN_TXT} \
    -o data/eng_train.txt \
    --entities {EN_TRAIN_ENT} \
    --neg-ratio {NEG_RATIO} \
    --rel2id data/rel2id.json

# English — dev split (positives only; negatives not needed for evaluation)
!python prepare_data.py {EN_DEV_REL} {EN_DEV_TXT} \
    -o data/eng_dev.txt \
    --entities {EN_DEV_ENT} \
    --neg-ratio {NEG_RATIO}

In [ ]:
# Russian — training split
!python prepare_data.py {RU_TRAIN_REL} {RU_TRAIN_TXT} \
    -o data/rus_train.txt \
    --entities {RU_TRAIN_ENT} \
    --neg-ratio {NEG_RATIO} \
    --lang russian

# Russian — dev split
!python prepare_data.py {RU_DEV_REL} {RU_DEV_TXT} \
    -o data/rus_dev.txt \
    --entities {RU_DEV_ENT} \
    --neg-ratio {NEG_RATIO} \
    --lang russian

In [ ]:
# Bilingual — concatenate EN+RU (Subtask 3)
!cat data/eng_train.txt data/rus_train.txt > data/bilingual_train.txt
!cat data/eng_dev.txt   data/rus_dev.txt   > data/bilingual_dev.txt
!wc -l data/eng_train.txt data/rus_train.txt data/bilingual_train.txt

## 3. Quick data sanity check

In [ ]:
import json
from bionne_dataset import BioNNEDataset, get_typed_marker_tokens, insert_typed_markers
from transformers import AutoTokenizer

# Show a sample instance with typed markers
with open("data/eng_train.txt") as f:
    sample = json.loads(f.readline())

marked = insert_typed_markers(
    sample["text"],
    *sample["h"]["pos"], sample["head_type"],
    *sample["t"]["pos"], sample["tail_type"],
)
print("Relation :", sample["relation"])
print("Marked   :", marked[:300], "...")
print("Nested   :", sample["h"]["pos"][0] <= sample["t"]["pos"][0] and
                    sample["t"]["pos"][1] <= sample["h"]["pos"][1])

## 4. Train English model (Subtask 1 — BiomedBERT)

In [ ]:
from bionne_train import train

best_f1_en = train(
    train_path="data/eng_train.txt",
    dev_path="data/eng_dev.txt",
    rel2id_path="data/rel2id.json",
    ckpt_path="outputs/biomedbert_en.pt",
    model_name=MODEL_EN,
    max_length=MAX_LEN,
    batch_size=BATCH_SIZE,
    lr=LR,
    epochs=EPOCHS,
    warmup_steps=WARMUP,
    use_class_weights=True,
    use_nesting_flag=True,
    seed=SEED,
    gdrive_dir=GDRIVE_DIR,
    resume=RESUME,
)
print(f"\nBest EN macro F1: {best_f1_en:.4f}")

## 5. Predict & evaluate English dev

In [ ]:
from bionne_predict import predict

predict(
    data_path="data/eng_dev.txt",
    ckpt_path="outputs/biomedbert_en.pt",
    output_path="outputs/eng_dev_pred.tsv",
)

In [ ]:
!python score.py --pred outputs/eng_dev_pred.tsv --gold {EN_DEV_REL}

## 6. Train Russian model (Subtask 2 — mDeBERTa)

In [ ]:
best_f1_ru = train(
    train_path="data/rus_train.txt",
    dev_path="data/rus_dev.txt",
    rel2id_path="data/rel2id.json",
    ckpt_path="outputs/mdeberta_ru.pt",
    model_name=MODEL_RU,
    max_length=MAX_LEN,
    batch_size=BATCH_SIZE,
    lr=LR,
    epochs=EPOCHS,
    warmup_steps=WARMUP,
    use_class_weights=True,
    use_nesting_flag=True,
    seed=SEED,
    gdrive_dir=GDRIVE_DIR,
    resume=RESUME,
)
print(f"\nBest RU macro F1: {best_f1_ru:.4f}")

In [ ]:
predict(
    data_path="data/rus_dev.txt",
    ckpt_path="outputs/mdeberta_ru.pt",
    output_path="outputs/rus_dev_pred.tsv",
)

!python score.py --pred outputs/rus_dev_pred.tsv --gold {RU_DEV_REL}

## 7. (Optional) Train bilingual model (Subtask 3 — mDeBERTa)

In [ ]:
best_f1_bili = train(
    train_path="data/bilingual_train.txt",
    dev_path="data/bilingual_dev.txt",
    rel2id_path="data/rel2id.json",
    ckpt_path="outputs/mdeberta_bilingual.pt",
    model_name=MODEL_BILI,
    max_length=MAX_LEN,
    batch_size=BATCH_SIZE,
    lr=LR,
    epochs=EPOCHS,
    warmup_steps=WARMUP,
    use_class_weights=True,
    use_nesting_flag=True,
    seed=SEED,
    gdrive_dir=GDRIVE_DIR,
    resume=RESUME,
)
print(f"\nBest Bilingual macro F1: {best_f1_bili:.4f}")

In [ ]:
predict(
    data_path="data/bilingual_dev.txt",
    ckpt_path="outputs/mdeberta_bilingual.pt",
    output_path="outputs/bilingual_dev_pred.tsv",
)

!python score.py --pred outputs/bilingual_dev_pred.tsv --gold {RU_DEV_REL}

## 8. Final test-set predictions (blind — for CodaBench submission)

Test data is released (`data/en/test/` and `data/ru/test/`). Run the cells below in order:
1. Restore checkpoints from Drive (or skip if already in `outputs/`)
2. Subtask 1 — English predictions → `outputs/eng_test_pred.tsv`
3. Subtask 2 — Russian predictions → `outputs/rus_test_pred.tsv`
4. Subtask 3 — Bilingual predictions → `outputs/bilingual_test_pred.tsv`
5. Save all prediction TSVs back to Drive

## 8a. Threshold calibration — MUST run before test predictions

**Why this matters:** Dev evaluation used gold pairs only (~12K instances). Test uses ALL ordered
entity pairs (~818K for EN, ~664K for RU). The model was trained with neg_ratio=3 (~25% positive
rate) but the true test positive rate is ~1%. Without a threshold, ~200K false positive predictions
are submitted → precision ~4% → test F1 collapses.

Run the cells below to find the confidence threshold that maximises blind-mode dev macro F1.
Use that threshold in Section 8 test predictions.

In [ ]:
# ── Prepare dev in BLIND mode (entity TSV → all pairs, same as test) ──────
# This is the correct way to evaluate before submitting.
!python prepare_data.py {EN_DEV_ENT} {EN_DEV_TXT} -o data/eng_dev_blind.txt
!python prepare_data.py {RU_DEV_ENT} {RU_DEV_TXT} -o data/rus_dev_blind.txt --lang russian
!cat data/eng_dev_blind.txt data/rus_dev_blind.txt > data/bilingual_dev_blind.txt
!wc -l data/eng_dev_blind.txt data/rus_dev_blind.txt data/bilingual_dev_blind.txt

In [ ]:
# ── EN threshold calibration — coarse then fine sweep ─────────────────────
import importlib, bionne_predict, bionne_dataset
importlib.reload(bionne_dataset)
importlib.reload(bionne_predict)
from bionne_predict import _run_inference, _probs_to_df
from score import evaluate, load_gold
from pathlib import Path

# Single inference pass — dataset pre-tokenises all instances in batch mode at init
en_instances, en_probs, en_rel2id = _run_inference(
    "data/eng_dev_blind.txt",
    "outputs/biomedbert_en.pt",
    batch_size=256,
    num_workers=0,
)
gold_en = load_gold(Path(EN_DEV_REL))

print(f"\n{'Threshold':>10}  {'#Preds':>8}  {'MacroF1':>8}  {'Precision':>10}  {'Recall':>10}")
print("-" * 58)
best_f1_en, THRESHOLD_EN = -1.0, 0.0
for t in [i / 100 for i in range(0, 100, 10)] + [i / 100 for i in range(90, 100)]:
    pred_df = _probs_to_df(en_instances, en_probs, en_rel2id, t)
    r = evaluate(pred_df, gold_en)
    marker = "  ◄" if r["macro_f1"] > best_f1_en else ""
    print(f"{t:>10.2f}  {len(pred_df):>8,}  {r['macro_f1']:>8.4f}"
          f"  {r['micro_precision']:>10.4f}  {r['micro_recall']:>10.4f}{marker}")
    if r["macro_f1"] > best_f1_en:
        best_f1_en, THRESHOLD_EN = r["macro_f1"], t

print(f"\nBest EN threshold: {THRESHOLD_EN}  (blind dev macro F1 = {best_f1_en:.4f})")

In [ ]:
# ── RU + Bilingual threshold calibration ──────────────────────────────────
import pandas as pd
from pathlib import Path

THRESHOLDS = [i / 100 for i in range(0, 100, 10)] + [i / 100 for i in range(90, 100)]

# ── RU ──
ru_instances, ru_probs, ru_rel2id = _run_inference(
    "data/rus_dev_blind.txt", "outputs/mdeberta_ru.pt",
    batch_size=256, num_workers=0,
)
gold_ru = load_gold(Path(RU_DEV_REL))

print("Russian:")
print(f"{'Threshold':>10}  {'#Preds':>8}  {'MacroF1':>8}  {'Precision':>10}  {'Recall':>10}")
print("-" * 58)
best_f1_ru, THRESHOLD_RU = -1.0, 0.0
for t in THRESHOLDS:
    pred_df = _probs_to_df(ru_instances, ru_probs, ru_rel2id, t)
    r = evaluate(pred_df, gold_ru)
    marker = "  ◄" if r["macro_f1"] > best_f1_ru else ""
    print(f"{t:>10.2f}  {len(pred_df):>8,}  {r['macro_f1']:>8.4f}"
          f"  {r['micro_precision']:>10.4f}  {r['micro_recall']:>10.4f}{marker}")
    if r["macro_f1"] > best_f1_ru:
        best_f1_ru, THRESHOLD_RU = r["macro_f1"], t

# ── Bilingual (gold = EN dev + RU dev combined) ──
bili_gold_path = "data/bilingual_dev_gold.tsv"
pd.concat([
    pd.read_csv(EN_DEV_REL, sep="\t"),
    pd.read_csv(RU_DEV_REL, sep="\t"),
]).to_csv(bili_gold_path, sep="\t", index=False)
gold_bili = load_gold(Path(bili_gold_path))

bili_instances, bili_probs, bili_rel2id = _run_inference(
    "data/bilingual_dev_blind.txt", "outputs/mdeberta_bilingual.pt",
    batch_size=256, num_workers=0,
)

print("\nBilingual:")
print(f"{'Threshold':>10}  {'#Preds':>8}  {'MacroF1':>8}  {'Precision':>10}  {'Recall':>10}")
print("-" * 58)
best_f1_bili, THRESHOLD_BILI = -1.0, 0.0
for t in THRESHOLDS:
    pred_df = _probs_to_df(bili_instances, bili_probs, bili_rel2id, t)
    r = evaluate(pred_df, gold_bili)
    marker = "  ◄" if r["macro_f1"] > best_f1_bili else ""
    print(f"{t:>10.2f}  {len(pred_df):>8,}  {r['macro_f1']:>8.4f}"
          f"  {r['micro_precision']:>10.4f}  {r['micro_recall']:>10.4f}{marker}")
    if r["macro_f1"] > best_f1_bili:
        best_f1_bili, THRESHOLD_BILI = r["macro_f1"], t

print(f"\nThresholds to use in Section 8:")
print(f"  EN:        {THRESHOLD_EN}  (blind dev F1 = {best_f1_en:.4f})")
print(f"  RU:        {THRESHOLD_RU}  (blind dev F1 = {best_f1_ru:.4f})")
print(f"  Bilingual: {THRESHOLD_BILI}  (blind dev F1 = {best_f1_bili:.4f})")

In [ ]:
# ── Restore checkpoints from Google Drive ────────────────────────────────
import shutil, os
from google.colab import drive
drive.mount("/content/drive")
os.makedirs("outputs", exist_ok=True)
os.makedirs("data", exist_ok=True)

for fname in ["biomedbert_en.pt", "mdeberta_ru.pt", "mdeberta_bilingual.pt"]:
    src = os.path.join(GDRIVE_DIR, fname)
    dest = os.path.join("outputs", fname)
    if os.path.exists(src):
        shutil.copy2(src, dest)
        print(f"  restored  {fname}  ({os.path.getsize(dest)/1e6:.0f} MB)")
    else:
        print(f"  NOT FOUND  {src}  — run Section 10 first to save from a training session")

for fname in ["rel2id.json"]:
    src = os.path.join(GDRIVE_DIR, fname)
    dest = os.path.join("data", fname)
    if os.path.exists(src):
        shutil.copy2(src, dest)
        print(f"  restored  {fname}")
    else:
        print(f"  NOT FOUND  {src}")

In [ ]:
!pip install -U nltk

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 5.7 MB/s eta 0:00:00:00:0100:01
  Attempting uninstall: nltk
    Found existing installation: nltk 3.8.1
    Uninstalling nltk-3.8.1:
      Successfully uninstalled nltk-3.8.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cognee 0.2.3 requires openai<1.99.9,>=1.80.1, but you have openai 1.109.1 which is incompatible.


In [ ]:
import nltk

nltk.download("punkt_tab")

[nltk_data] Downloading package punkt_tab to /home/obi/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [ ]:
# ── Subtask 1: English test predictions (BiomedBERT) ─────────────────────
import importlib, bionne_predict
importlib.reload(bionne_predict)
from bionne_predict import predict

EN_TEST_ENT = f"{TASK_DIR}/data/en/test/eng-test-ent.tsv"
EN_TEST_TXT = f"{TASK_DIR}/data/en/test/texts/"

!python prepare_data.py {EN_TEST_ENT} {EN_TEST_TXT} -o data/eng_test.txt

THRESHOLD_EN = 0.99   # calibrated via blind dev sweep

predict(
    data_path="data/eng_test.txt",
    ckpt_path="outputs/biomedbert_en.pt",
    output_path="outputs/eng_test_pred.tsv",
    threshold=THRESHOLD_EN,
)

In [ ]:
from bionne_predict import predict
predict(
    data_path="data/eng_test.txt",
    ckpt_path="outputs/biomedbert_en.pt",
    output_path="outputs/eng_test_pred.tsv",
)

In [ ]:
# ── Subtask 2: Russian test predictions (mDeBERTa) ────────────────────────
RU_TEST_ENT = f"{TASK_DIR}/data/ru/test/rus-test-ent.tsv"
RU_TEST_TXT = f"{TASK_DIR}/data/ru/test/texts/"

!python prepare_data.py {RU_TEST_ENT} {RU_TEST_TXT} -o data/rus_test.txt --lang russian

Wrote data/rel2id.json (15 classes)
Detected input type: entity
Blind mode: ../data/ru/test/rus-test-ent.tsv + ../data/ru/test/texts -> data/rus_test.txt
  664448 candidate pairs
Done.


In [ ]:
# ── Subtask 2: Russian test predictions (mDeBERTa) ────────────────────────
THRESHOLD_RU = 0.99   # calibrated via blind dev sweep (trend matches EN)

RU_TEST_ENT = f"{TASK_DIR}/data/ru/test/rus-test-ent.tsv"
RU_TEST_TXT = f"{TASK_DIR}/data/ru/test/texts/"

!python prepare_data.py {RU_TEST_ENT} {RU_TEST_TXT} -o data/rus_test.txt --lang russian

predict(
    data_path="data/rus_test.txt",
    ckpt_path="outputs/mdeberta_ru.pt",
    output_path="outputs/rus_test_pred.tsv",
    threshold=THRESHOLD_RU,
)

In [ ]:
# ── Subtask 3: Bilingual test predictions (mDeBERTa, EN+RU combined) ──────
# Requires eng_test.txt and rus_test.txt — run Subtask 1 & 2 cells first.
!cat data/eng_test.txt data/rus_test.txt > data/bilingual_test.txt
!wc -l data/eng_test.txt data/rus_test.txt data/bilingual_test.txt

    818084 data/eng_test.txt
    664448 data/rus_test.txt
   1482532 data/bilingual_test.txt
   2965064 total


In [ ]:
# ── Subtask 3: Bilingual test predictions (mDeBERTa, EN+RU combined) ──────
THRESHOLD_BILI = 0.99   # calibrated via blind dev sweep (trend matches EN/RU)

# Requires eng_test.txt and rus_test.txt — run Subtask 1 & 2 cells first.
!cat data/eng_test.txt data/rus_test.txt > data/bilingual_test.txt
!wc -l data/eng_test.txt data/rus_test.txt data/bilingual_test.txt

predict(
    data_path="data/bilingual_test.txt",
    ckpt_path="outputs/mdeberta_bilingual.pt",
    output_path="outputs/bilingual_test_pred.tsv",
    threshold=THRESHOLD_BILI,
)

In [ ]:
# ── Save test predictions to Google Drive ────────────────────────────────
import shutil, os, pandas as pd
for fname in ["eng_test_pred.tsv", "rus_test_pred.tsv", "bilingual_test_pred.tsv"]:
    src = f"outputs/{fname}"
    if os.path.exists(src):
        n = len(pd.read_csv(src, sep="\t"))
        shutil.copy2(src, os.path.join(GDRIVE_DIR, fname))
        print(f"  saved  {fname}  ({n} predictions)")
    else:
        print(f"  missing  {src}  — run the prediction cell above first")

## 9. Ablation: incorporate LLM-augmented data

After running `augment.py` locally (step 4 in the plan), append augmented
JSON-lines to the training files and retrain.

In [ ]:
# ── EN + augmented data ───────────────────────────────────────────────────
# After running augment.py locally and committing augmented_en.jsonl:
#
# !cat data/eng_train.txt data/augmented_en.jsonl > data/eng_train_aug.txt
#
# best_f1_en_aug = train(
#     train_path="data/eng_train_aug.txt",
#     dev_path="data/eng_dev.txt",
#     rel2id_path="data/rel2id.json",
#     ckpt_path="outputs/biomedbert_en_aug.pt",   # separate file — does NOT overwrite biomedbert_en.pt
#     model_name=MODEL_EN,
#     max_length=MAX_LEN, batch_size=BATCH_SIZE, lr=LR,
#     epochs=EPOCHS, warmup_steps=WARMUP,
#     use_class_weights=True, use_nesting_flag=True, seed=SEED,
#     gdrive_dir=GDRIVE_DIR,
# )
# print(f"EN aug macro F1:      {best_f1_en_aug:.4f}  (no-aug: {best_f1_en:.4f})")

# ── RU + augmented data ───────────────────────────────────────────────────
# After running augment.py on the RU training data:
#
# !cat data/rus_train.txt data/augmented_ru.jsonl > data/rus_train_aug.txt
#
# best_f1_ru_aug = train(
#     train_path="data/rus_train_aug.txt",
#     dev_path="data/rus_dev.txt",
#     rel2id_path="data/rel2id.json",
#     ckpt_path="outputs/mdeberta_ru_aug.pt",     # separate file — does NOT overwrite mdeberta_ru.pt
#     model_name=MODEL_RU,
#     max_length=MAX_LEN, batch_size=BATCH_SIZE, lr=LR,
#     epochs=EPOCHS, warmup_steps=WARMUP,
#     use_class_weights=True, use_nesting_flag=True, seed=SEED,
#     gdrive_dir=GDRIVE_DIR,
# )
# print(f"RU aug macro F1:      {best_f1_ru_aug:.4f}  (no-aug: {best_f1_ru:.4f})")

## 10. Save checkpoints to Google Drive

Run this cell after training finishes (or after any individual training cell).
Checkpoints are saved under `MyDrive/BioNNE-R/` and can be loaded directly on
a fresh Colab session without retraining.

In [ ]:
import os
import shutil
import torch
from google.colab import drive

drive.mount("/content/drive")
os.makedirs(GDRIVE_DIR, exist_ok=True)

CHECKPOINTS = {
    "outputs/biomedbert_en.pt":      "biomedbert_en.pt",
    "outputs/biomedbert_en_aug.pt":  "biomedbert_en_aug.pt",
    "outputs/mdeberta_ru.pt":        "mdeberta_ru.pt",
    "outputs/mdeberta_ru_aug.pt":    "mdeberta_ru_aug.pt",
    "outputs/mdeberta_bilingual.pt": "mdeberta_bilingual.pt",
}

for local_path, drive_name in CHECKPOINTS.items():
    if not os.path.exists(local_path):
        print(f"  skip  {local_path}  (not found)")
        continue
    size_mb = os.path.getsize(local_path) / 1e6
    try:
        ckpt = torch.load(local_path, map_location="cpu")
        f1   = ckpt.get("macro_f1", float("nan"))
        ep   = ckpt.get("epoch", "?")
    except Exception as e:
        print(f"  CORRUPT  {local_path}  ({size_mb:.1f} MB) — {e}")
        print(f"           Delete it and retrain to get a valid checkpoint.")
        continue
    dest = os.path.join(GDRIVE_DIR, drive_name)
    shutil.copy2(local_path, dest)
    print(f"  saved  {drive_name}  ({size_mb:.1f} MB, epoch {ep}, macro F1 {f1:.4f})")

    # Also copy the .resume file so training can be resumed on a fresh session.
    resume_path = local_path + ".resume"
    if os.path.exists(resume_path):
        resume_state = torch.load(resume_path, map_location="cpu")
        shutil.copy2(resume_path, os.path.join(GDRIVE_DIR, drive_name + ".resume"))
        print(f"         + resume file (epoch {resume_state.get('epoch', '?')})")

shutil.copy2("data/rel2id.json", os.path.join(GDRIVE_DIR, "rel2id.json"))
print(f"\nAll files saved to: {GDRIVE_DIR}")